# Phase 5 - Word2Vec + KMeans Clustering

Goal: train Word2Vec embeddings on the 311 corpus, average per document, run K-Means with k swept over [5, 30], pick the best k by silhouette, and surface top terms per cluster.

This is the *latent issue discovery* objective from the proposal. Where Phase 3's classifier learns to predict the official taxonomy, K-Means clusters tell us what groupings *would* exist if the city redrew the taxonomy from scratch. Often clusters span multiple official categories - rats + trash + vacant lot all collapse into one 'urban decay' cluster.

Phases 0-2 must be passing. Phase 2's `sample_2m_preprocessed.parquet` must be on Drive.

## Cell 1 - Bootstrap

In [ ]:
REPO_URL = 'https://github.com/george-gideon-S/cs-gy-6513-big-data-311-nlp.git'

from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys
if not os.path.isdir('/content/project/.git'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/project'], check=True)
else:
    subprocess.run(['git', '-C', '/content/project', 'pull'], check=True)

if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')

!pip install -r /content/project/requirements.txt -q

!apt-get install -y openjdk-11-jre-headless > /dev/null 2>&1
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

import nltk
for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    nltk.download(pkg, download_dir='/root/nltk_data', quiet=True)

from src.spark_setup import get_spark
spark = get_spark(app_name='phase5-word2vec')
print('spark', spark.version, 'ready')

## Cell 2 - Load preprocessed data + filter

Word2Vec needs at least one token per row. We drop empty rows and require at least 2 tokens for clustering quality (single-token rows produce nearly identical doc vectors which make silhouette meaningless).

In [ ]:
from pyspark.sql import functions as F

in_path = '/content/drive/MyDrive/cs6513/sample_2m_preprocessed.parquet'
df = spark.read.parquet(in_path).filter(F.size('tokens') >= 2)
n = df.count()
print(f'rows with >=2 tokens: {n:,}')

## Cell 3 - Train Word2Vec

vector size 100, window 5, min count 5. With ~1.9M short docs the training takes 2-4 min on Colab High-RAM. Word2Vec's MLlib implementation is single-threaded skip-gram so its actually CPU-bound, not memory-bound.

In [ ]:
from pyspark.ml.feature import Word2Vec
import time

w2v = Word2Vec(
    vectorSize=100,
    windowSize=5,
    minCount=5,
    inputCol='tokens',
    outputCol='doc_vec',
    seed=42,
)

t0 = time.time()
w2v_model = w2v.fit(df)
t_fit_w2v = time.time() - t0
print(f'word2vec fit in {t_fit_w2v:.1f} sec')

vocab_size = w2v_model.getVectors().count()
print(f'learned vocabulary size: {vocab_size:,}')

## Cell 4 - Inspect Word2Vec quality (find synonyms for known words)

Quick sanity check before clustering. If 'rat' returns ['rodent', 'mouse', 'pest'] we know the embeddings are sensible.

In [ ]:
probe_words = ['rat', 'noise', 'pothole', 'leak', 'graffiti', 'tree']
print('top 5 nearest words for probe terms (cosine similarity):')
for w in probe_words:
    try:
        synonyms = w2v_model.findSynonymsArray(w, 5)
        formatted = ', '.join(f'{s[0]} ({s[1]:.2f})' for s in synonyms)
        print(f'  {w:12s} -> {formatted}')
    except Exception as e:
        print(f'  {w:12s} -> not in vocab ({e})')

## Cell 5 - Apply Word2Vec to get doc vectors

Spark's Word2Vec.transform() averages the word vectors per row, producing one fixed-dim embedding per document. We cache so the K-Means sweep doesn't recompute every iteration.

In [ ]:
df_vec = w2v_model.transform(df).select('unique_key', 'label_canonical', 'tokens', 'doc_vec').cache()
print(f'doc_vec computed; cached for kmeans sweep')
df_vec.show(3, truncate=60)

## Cell 6 - K-Means sweep (k = 5, 10, 15, 20, 25, 30)

Fit K-Means at each k, evaluate by silhouette. We pick the k with the highest silhouette score for the final clustering. Each fit is 30-90 sec; the full sweep takes 5-10 min.

We sample 200K rows for the silhouette evaluator since it's O(n^2) on each cluster (full 1.9M would take an hour just to score).

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# the predictionCol='cluster' here matches what KMeans writes - if you leave
# it at the default 'prediction', the evaluator throws IllegalArgumentException
evaluator = ClusteringEvaluator(
    featuresCol='doc_vec',
    predictionCol='cluster',
    metricName='silhouette',
)

# sample for evaluation (silhouette is O(n^2))
df_sample_for_eval = df_vec.sample(fraction=200_000 / n, seed=42).cache()
df_sample_for_eval.count()  # materialize

results = []
for k in [5, 10, 15, 20, 25, 30]:
    t0 = time.time()
    km = KMeans(featuresCol='doc_vec', predictionCol='cluster', k=k, seed=42, maxIter=20)
    km_model = km.fit(df_sample_for_eval)
    t_fit = time.time() - t0

    preds = km_model.transform(df_sample_for_eval)
    score = evaluator.evaluate(preds)
    results.append((k, score, t_fit, km_model))
    print(f'  k={k:>3}  silhouette={score:.4f}  fit_time={t_fit:.1f}s')

best_k, best_score, _, best_model = max(results, key=lambda r: r[1])
print(f'\nbest k = {best_k} (silhouette = {best_score:.4f})')

## Cell 7 - Plot silhouette curve

Saves to `dashboard/assets/silhouette_curve.png` for the Cluster Atlas tab.

In [ ]:
import matplotlib.pyplot as plt

ks = [r[0] for r in results]
scores = [r[1] for r in results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ks, scores, marker='o', linewidth=2, color='#57068C')
ax.scatter([best_k], [best_score], s=200, color='#FF6F00', zorder=5, label=f'best k={best_k}')
ax.set_xlabel('k')
ax.set_ylabel('silhouette score')
ax.set_title('K-Means silhouette by k (Word2Vec doc embeddings)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('/content/project/dashboard/assets/silhouette_curve.png', dpi=120)
plt.show()
print('saved silhouette_curve.png')

## Cell 8 - Refit best K-Means on full corpus + assign clusters

The sweep used a 200K sample for silhouette. Now refit on the full data so every row gets a cluster assignment.

In [ ]:
km_final = KMeans(featuresCol='doc_vec', predictionCol='cluster', k=best_k, seed=42, maxIter=20)
t0 = time.time()
km_final_model = km_final.fit(df_vec)
print(f'final kmeans fit in {time.time()-t0:.1f} sec')

df_clustered = km_final_model.transform(df_vec)

# cluster size distribution
cluster_sizes = (
    df_clustered.groupBy('cluster').count()
    .orderBy('cluster').toPandas()
)
print('\ncluster sizes:')
print(cluster_sizes.to_string(index=False))

## Cell 9 - Top terms per cluster (the qualitative story)

For each cluster, find the most-frequent tokens. This is the human-readable answer to 'what is each cluster about?'.

In [ ]:
import pandas as pd

# explode + count tokens per cluster
exploded = df_clustered.select('cluster', F.explode('tokens').alias('term'))
term_counts = exploded.groupBy('cluster', 'term').count().toPandas()

cluster_top_terms = {}
for cluster_id in range(best_k):
    sub = term_counts[term_counts['cluster'] == cluster_id]
    top = sub.nlargest(10, 'count')
    cluster_top_terms[int(cluster_id)] = list(zip(top['term'].tolist(), top['count'].astype(int).tolist()))

print('top 10 terms per cluster:')
for cid, terms in cluster_top_terms.items():
    formatted = ', '.join(f'{t}({c})' for t, c in terms[:6])  # only show 6 in print to keep readable
    print(f'  cluster {cid:>2}: {formatted}')

## Cell 10 - Cluster vs. official category cross-tab

Where do clusters and the city's official categories agree, and where do they diverge? This is the 'latent issue discovery' result.

In [ ]:
cross = (
    df_clustered.groupBy('cluster', 'label_canonical').count()
    .toPandas()
)

# for each cluster, what are the top 3 official categories represented?
cluster_categories = {}
for cid in range(best_k):
    sub = cross[cross['cluster'] == cid].nlargest(3, 'count')
    total = sub['count'].sum()
    cluster_categories[int(cid)] = [
        (row['label_canonical'], int(row['count']), round(100 * row['count'] / total, 1))
        for _, row in sub.iterrows()
    ]

print(f'top 3 official categories per cluster (with % of cluster):')
for cid, cats in cluster_categories.items():
    formatted = '  |  '.join(f'{name} ({pct}%)' for name, _cnt, pct in cats)
    print(f'  cluster {cid:>2}: {formatted}')

## Cell 11 - Save artifacts

Three artifacts:
1. **Word2Vec model on Drive** for Phase 7 dashboard.
2. **Gensim KeyedVectors** in repo - portable, deployable, no Spark needed.
3. **Cluster summary JSON** with top terms + cross-tab for the dashboard's Cluster Atlas tab.

In [ ]:
# 1. word2vec spark model on drive
w2v_path = '/content/drive/MyDrive/cs6513/models/word2vec'
w2v_model.write().overwrite().save(w2v_path)
print(f'word2vec spark model saved to {w2v_path}')

# 2. portable gensim KeyedVectors
import gensim, numpy as np
vec_pdf = w2v_model.getVectors().toPandas()
words = vec_pdf['word'].tolist()
vectors = np.stack([v.toArray() for v in vec_pdf['vector']])

kv = gensim.models.KeyedVectors(vector_size=vectors.shape[1])
kv.add_vectors(words, vectors)

kv_path = '/content/project/models/portable/word2vec.kv'
os.makedirs(os.path.dirname(kv_path), exist_ok=True)
kv.save(kv_path)
size_mb = os.path.getsize(kv_path) / 1024 / 1024
print(f'gensim KeyedVectors saved to {kv_path} ({size_mb:.2f} MB)')

# 3. cluster summary json
import json, datetime
summary = {
    'phase': 5,
    'trained_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'n_docs': int(n),
    'vocab_size': int(vocab_size),
    'word2vec': {'vector_size': 100, 'window': 5, 'min_count': 5},
    'kmeans': {'best_k': int(best_k), 'silhouette': float(best_score)},
    'sweep': [{'k': int(r[0]), 'silhouette': float(r[1])} for r in results],
    'cluster_top_terms': {str(k): v for k, v in cluster_top_terms.items()},
    'cluster_categories': {str(k): v for k, v in cluster_categories.items()},
    'training_time_sec': {'word2vec': float(t_fit_w2v)},
}
with open('/content/project/dashboard/assets/cluster_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('saved cluster_summary.json')

## Cell 12 - Push artifacts to GitHub

Commits the word2vec.kv portable + cluster_summary.json + silhouette curve png to the repo. Requires `GITHUB_PAT` in Colab Secrets.

In [ ]:
from src.colab_git import commit_artifacts
commit_artifacts(message='phase 5: word2vec + clustering artifacts')

## Phase 5 - Done when

- Cell 4 returns sensible synonyms (e.g., `rat -> rodent, mouse, mice`).
- Cell 6 produces a silhouette curve with at least one positive score (clusters are real, not noise).
- Cell 9 prints top-10 terms per cluster that look human-interpretable.
- Cell 10 shows interesting cross-cluster spans (e.g., one cluster contains both Rodent and Dirty Conditions categories - the 'urban decay' cluster from the proposal).
- Cell 11 saves Word2Vec + KeyedVectors + cluster_summary.json.

Save the print as `PRINT 6.pdf` and drop in the project directory. Then we move to Phase 6 (geographic + census join).